# 05 — SHAP Explainability

SHAP (SHapley Additive exPlanations) tells us WHICH words/features
caused the model to classify a post as fake.

This powers the 'Reason Cards' on the frontend.
We apply SHAP to the Random Forest model (faster than BERT for inference).

In [1]:
import shap
import joblib
import numpy as np

# Load saved model and vectorizer
rf = joblib.load('../backend/models/rf_model.pkl')
tfidf = joblib.load('../backend/models/tfidf_vectorizer.pkl')
print('Model and vectorizer loaded!')

Model and vectorizer loaded!


In [2]:
# Create SHAP explainer
explainer = shap.TreeExplainer(rf)
print('SHAP explainer created!')

SHAP explainer created!


In [3]:
# Test on a sample fake post
sample_text = 'Work from home earn 50000 per month no experience needed apply now cash daily'
sample_vec = tfidf.transform([sample_text]).toarray()
shap_values = explainer.shap_values(sample_vec, check_additivity=False)

# Handle both old (list) and new (3D array) SHAP output formats
if isinstance(shap_values, list):
    shap_for_fake = shap_values[1][0]  # Old SHAP: list of arrays per class
else:
    shap_for_fake = shap_values[0, :, 1]  # New SHAP: 3D array (sample, feature, class)

# Get top contributing features (words)
feature_names = tfidf.get_feature_names_out()
top_indices = np.argsort(np.abs(shap_for_fake))[::-1][:10]
top_words = [(feature_names[i], shap_for_fake[i]) for i in top_indices]
print('Top words causing fake classification:')
for word, val in top_words:
    print(f'  {word}: {val:.4f}')
# This output becomes the Reason Cards on frontend

Top words causing fake classification:
  across: -87599279774365152521513615753216.0000
  startups: -67198362080757608607708912549888.0000
  crm: -64414460830885100175791652601856.0000
  innovative: -60395805184508880600149390262272.0000
  fast growing: -58473678280602598618288093134848.0000
  services: -54481633734157327598465279066112.0000
  products and: -51351242475232097967848353169408.0000
  leaders: -50050711735033095662035420577792.0000
  equal: -48528473283238132098231653367808.0000
  work closely: -44113914146026337540559840739328.0000


In [4]:
# Test on a real job post
real_text = ('Software Engineer at Google. We are looking for an experienced '
             'software engineer to join our team. Requirements: 3+ years of '
             'experience in Python, Java, or C++. BS in Computer Science. '
             'Benefits: Health insurance, 401k, flexible hours.')
real_vec = tfidf.transform([real_text]).toarray()
real_shap = explainer.shap_values(real_vec, check_additivity=False)

# Handle both old and new SHAP output formats
if isinstance(real_shap, list):
    real_shap_vals = real_shap[1][0]
else:
    real_shap_vals = real_shap[0, :, 1]

real_top = np.argsort(np.abs(real_shap_vals))[::-1][:10]
print('Top words for REAL post:')
for i in real_top:
    print(f'  {feature_names[i]}: {real_shap_vals[i]:.4f}')

Top words for REAL post:
  fast growing: 9334183212162083404428127240192.0000
  innovative: 8750506660681468878503844773888.0000
  digital: 8282630260800731710054801080320.0000
  services: 7403224468633059354756975165440.0000
  the day: 6440461462873745440481990410240.0000
  interviews: 6177110643035217652436388806656.0000
  startups: 6050046277692476723158734340096.0000
  the country: 5867220725390376464948110819328.0000
  of highly: 5783727635941922548673425702912.0000
  sign: 5767667931901048756650914086912.0000


In [5]:
# Save explainer
joblib.dump(explainer, '../backend/models/shap_explainer.pkl')
print('SHAP explainer saved to backend/models/shap_explainer.pkl')

SHAP explainer saved to backend/models/shap_explainer.pkl
